In [ ]:
import os
import warnings
warnings.filterwarnings('ignore')
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'

import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.datasets import mnist

(x_train, y_train), (x_test, y_test) = mnist.load_data()

# Normalizacja danych
x_train, x_test = x_train / 255.0, x_test / 255.0

In [ ]:
#ile czasu zajmuje jeden krok dla poszczególnych wartości batch size?
batch_sizes = [8, 16, 32, 64, 128, 256]
histories = []

for batch_size in batch_sizes:
    print(f"Ustawiona wielość zbioru: {batch_size}")
    model = Sequential([
        Flatten(input_shape=(28, 28, 1)),
        Dense(128, activation='relu'),
        Dense(10, activation='softmax')
    ])

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    history = model.fit(x_train, y_train, epochs=5, batch_size=batch_size, validation_data=(x_test, y_test))
    histories.append(history)

In [ ]:
rows = []
for bsz, hist in zip(batch_sizes, histories):
    df = pd.DataFrame(hist.history)             
    df['epoch'] = np.arange(1, len(df) + 1)      
    df['batch_size'] = bsz
    # uporządkuj kolumny:
    df = df[['batch_size', 'epoch'] + [c for c in df.columns if c not in ['batch_size', 'epoch']]]
    rows.append(df)

full = pd.concat(rows, ignore_index=True)
full.to_csv('mnist_histories.csv', index=False)
print('Zapisano do mnist_histories.csv')